[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/advanced-ab-testing/blob/main/assignments/week08_assignment.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 8 Assignment: Sequential Testing (Wald's SPRT)

## QuickCart — Making Faster Experiment Decisions

QuickCart's experiments currently run for a fixed duration (usually 2-4 weeks). But sometimes the evidence is overwhelming after just a few days — a change either clearly helps or clearly hurts. The product team wants to know: **can we stop experiments early when the evidence is clear, without inflating false positive rates?**

The answer is yes — using **Sequential Testing**. Specifically, you'll implement **Wald's Sequential Probability Ratio Test (SPRT)**, which:
- Monitors evidence as data accumulates
- Stops as soon as there's enough evidence to decide (reject H0 or accept H0)
- Controls both Type I and Type II error rates at pre-specified levels

**But first, you need to understand WHY you can't just peek at a normal test repeatedly.**

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

np.random.seed(42)

---

## Task 1: The Peeking Problem

### Context

A common mistake: run an A/B test, check daily if it's significant, and stop as soon as p < 0.05. This seems reasonable but dramatically inflates false positive rates.

**Why?** Each peek is an independent chance to cross the significance threshold by luck. Over 30 days of peeking, the probability of *ever* crossing 0.05 is much higher than 5%.

### Problem

Demonstrate the peeking problem:

1. Simulate 1000 AA tests (no true effect) where data arrives daily for 30 days
2. Each day, 100 new observations per group arrive
3. After each day's data arrives, run a t-test on ALL accumulated data
4. Track whether the test was EVER significant (p < 0.05) during the 30-day window

Report: What fraction of the 1000 null experiments would be incorrectly declared significant if we peek daily?

**Expected result:** ~20-25% false positive rate (vs. the nominal 5%)

<details>
<summary>Hint (click to expand)</summary>

```python
n_simulations = 1000
n_days = 30
observations_per_day = 100

ever_significant = 0

for sim in range(n_simulations):
    control_data = []
    treatment_data = []
    was_significant = False
    
    for day in range(n_days):
        # Add new day's data (both from same distribution = no effect)
        control_data.extend(np.random.normal(0, 1, observations_per_day))
        treatment_data.extend(np.random.normal(0, 1, observations_per_day))
        
        # Peek: run t-test on all accumulated data
        _, p_value = stats.ttest_ind(control_data, treatment_data)
        if p_value < 0.05:
            was_significant = True
            break  # stop peeking (we'd declare significance)
    
    if was_significant:
        ever_significant += 1

false_positive_rate = ever_significant / n_simulations
```

</details>

In [ ]:
# YOUR CODE HERE
#
# Simulate the peeking problem:
# 1. Run 1000 AA tests with daily peeking
# 2. Report the inflated false positive rate
# 3. (Optional) Also track how the cumulative FPR grows with number of peeks

n_simulations = 1000
n_days = 30
observations_per_day = 100
alpha = 0.05

# YOUR CODE HERE
pass

In [ ]:
# Verification (uncomment after completing):

# print(f"Nominal alpha: {alpha}")
# print(f"Actual false positive rate with daily peeking: {false_positive_rate:.1%}")
# print(f"Inflation factor: {false_positive_rate / alpha:.1f}x")

# assert false_positive_rate > 0.15, \
#     f"Expected FPR > 15% with peeking, got {false_positive_rate:.1%}. Check your simulation."
# assert false_positive_rate < 0.40, \
#     f"FPR seems too high ({false_positive_rate:.1%}). Check your simulation."
# print("\nTask 1 check passed! Peeking inflates false positives as expected.")

In [ ]:
# Optional: Plot how false positive rate grows with number of peeks
# YOUR CODE HERE
#
# Suggested: For each day d in [1, 30], compute the fraction of simulations
# that ever crossed p < 0.05 by day d. Plot this curve.
pass

---

## Task 2: Implement the Sequential Tester (SPRT)

### Context

Wald's SPRT solves the peeking problem by using **different boundaries** that account for multiple looks at the data. Instead of comparing a p-value to a fixed threshold, it tracks the cumulative **log-likelihood ratio**:

$$\Lambda_n = \sum_{i=1}^n \frac{\delta}{\sigma^2}\left(x_i - \frac{\delta}{2}\right)$$

where $\delta$ is the effect size (H1) and $\sigma$ is the known standard deviation.

**Decision boundaries:**
- Upper boundary: $B = \log\frac{1-\beta}{\alpha}$ — if $\Lambda_n \geq B$, reject H0 (effect is real)
- Lower boundary: $A = \log\frac{\beta}{1-\alpha}$ — if $\Lambda_n \leq A$, reject H1 (no meaningful effect)
- If $A < \Lambda_n < B$, continue collecting data

### Problem

Implement the `SequentialTester` class matching the study notes API:

- `__init__(effect_size, alpha, beta, sigma)` — simple numeric parameters
- `add_observation(x)` — process one observation, check boundaries
- `add_batch(x_batch)` — process a batch of observations
- `run_test(data_stream, batch_size=1)` — process a full data stream, returns `(decision, n_observations)`

The input data $x_i$ represents the **difference** metric (e.g., treatment user value minus baseline).

<details>
<summary>Hint 1: Log-likelihood ratio per observation</summary>

For normal data with H0: $\mu=0$ and H1: $\mu=\delta$:

$$\log \frac{f_1(x)}{f_0(x)} = \frac{\delta}{\sigma^2}\left(x - \frac{\delta}{2}\right)$$

This is just arithmetic — no scipy distribution objects needed.

</details>

<details>
<summary>Hint 2: Expected drift</summary>

Under H0 (x ~ N(0, sigma^2)): the expected increment is $-\delta^2 / (2\sigma^2)$ (drifts toward lower boundary).

Under H1 (x ~ N(delta, sigma^2)): the expected increment is $+\delta^2 / (2\sigma^2)$ (drifts toward upper boundary).

</details>

In [ ]:
class SequentialTester:
    """Wald's Sequential Probability Ratio Test (SPRT) for normal data.
    
    Tests H0: mu = 0 vs H1: mu = effect_size
    using observations assumed to come from N(mu, sigma^2).
    
    Parameters
    ----------
    effect_size : float
        The minimum effect size to detect (mu under H1).
    alpha : float
        Type I error rate (probability of rejecting H0 when true).
    beta : float
        Type II error rate (probability of rejecting H1 when true).
    sigma : float
        Known (or estimated) standard deviation of observations.
    """
    
    def __init__(self, effect_size: float, alpha: float = 0.05, 
                 beta: float = 0.2, sigma: float = 1.0):
        # YOUR CODE HERE
        # Store parameters
        # Compute boundaries:
        #   upper_bound = log((1 - beta) / alpha)
        #   lower_bound = log(beta / (1 - alpha))
        # Initialize: cumulative_sum = 0, n_observations = 0, decision = None
        # Also store history = [] for visualization
        pass
    
    def _log_likelihood_ratio(self, x: float) -> float:
        """Compute log-likelihood ratio for a single observation.
        
        log(f1(x) / f0(x)) = (delta / sigma^2) * (x - delta/2)
        """
        # YOUR CODE HERE
        pass
    
    def add_observation(self, x: float):
        """Process a single observation and check boundaries.
        
        Returns
        -------
        str or None
            'reject_H0' if upper boundary crossed,
            'reject_H1' if lower boundary crossed,
            None if still in the continue zone.
        """
        # YOUR CODE HERE
        pass
    
    def add_batch(self, x_batch: np.ndarray):
        """Process a batch of observations.
        
        Returns decision as soon as a boundary is crossed.
        
        Returns
        -------
        str or None
            'reject_H0', 'reject_H1', or None.
        """
        # YOUR CODE HERE
        pass
    
    def run_test(self, data_stream: np.ndarray, batch_size: int = 1) -> tuple:
        """Process an entire data stream.
        
        Parameters
        ----------
        data_stream : array-like
            All observations to process sequentially.
        batch_size : int
            Number of observations per batch (default 1 = observation-level).
        
        Returns
        -------
        tuple
            (decision, n_observations) where decision is
            'reject_H0', 'reject_H1', or None (if data exhausted).
        """
        # YOUR CODE HERE
        pass
    
    def reset(self):
        """Reset the tester for a new experiment."""
        # YOUR CODE HERE
        pass

In [ ]:
# Test Task 2: Basic functionality

# Testing for a shift of 0.3 in mean (Normal data with sigma=1)
tester = SequentialTester(effect_size=0.3, alpha=0.05, beta=0.20, sigma=1.0)

# Check boundaries
expected_upper = np.log((1 - 0.20) / 0.05)  # log(0.8/0.05) = log(16) ~ 2.77
expected_lower = np.log(0.20 / (1 - 0.05))  # log(0.2/0.95) ~ -1.56

print(f"Upper bound: {tester.upper_bound:.4f} (expected ~{expected_upper:.4f})")
print(f"Lower bound: {tester.lower_bound:.4f} (expected ~{expected_lower:.4f})")

assert abs(tester.upper_bound - expected_upper) < 0.01, "Upper bound incorrect"
assert abs(tester.lower_bound - expected_lower) < 0.01, "Lower bound incorrect"

# Test with clear signal (strong effect under H1)
np.random.seed(42)
data_h1 = np.random.normal(0.5, 1.0, 500)  # effect larger than 0.3

decision, n_obs = tester.run_test(data_h1)
print(f"\nStrong signal test: decision='{decision}', stopped at observation {n_obs}")
assert decision == 'reject_H0', f"Should reject H0 with strong effect, got '{decision}'"
assert n_obs < 500, f"Should stop early with strong signal, used {n_obs} observations"

# Test with null data (should reject H1 = accept null)
tester.reset()
data_h0 = np.random.normal(0.0, 1.0, 500)  # no effect

decision, n_obs = tester.run_test(data_h0)
print(f"Null data test: decision='{decision}', stopped at observation {n_obs}")
assert decision == 'reject_H1', f"Should reject H1 (accept null) with no effect, got '{decision}'"

print("\nBasic Task 2 checks passed!")

In [ ]:
# Test batch processing (add_batch)
tester2 = SequentialTester(effect_size=0.3, alpha=0.05, beta=0.20, sigma=1.0)

np.random.seed(42)
# Add data in batches (simulating daily arrivals under H1)
for day in range(30):
    batch = np.random.normal(0.3, 1.0, 100)  # real effect = 0.3
    decision = tester2.add_batch(batch)
    if decision is not None:
        print(f"Decision made on day {day+1}: '{decision}'")
        print(f"Total observations processed: {tester2.n_observations}")
        break

if decision is None:
    print("No decision after 30 days")

assert decision == 'reject_H0', "Should detect the real effect"
print("\nBatch processing check passed!")

---

## Task 3: Validate Error Rate Control

### Context

The SPRT promises to control Type I error at $\alpha$ and Type II error at $\beta$. Verify this empirically.

### Problem

Run two sets of simulations:

**Under H0 (no effect):** Run 1000 simulations where data is N(0, sigma). Check that the false positive rate (fraction where decision is 'reject_H0') is at most $\alpha$.

**Under H1 (real effect):** Run 1000 simulations where data is N(effect_size, sigma). Check that the detection rate (fraction where decision is 'reject_H0') is at least $1 - \beta$.

Also report: What is the **average stopping time** compared to the fixed-horizon sample size? This demonstrates the speed advantage of sequential testing.

<details>
<summary>Hint</summary>

```python
n_simulations = 1000
effect_size = 0.3
sigma = 1.0
max_obs = 5000  # generous budget

decisions_h0 = []
n_obs_h0 = []

for sim in range(n_simulations):
    tester = SequentialTester(effect_size=effect_size, sigma=sigma)
    data = np.random.normal(0, sigma, max_obs)  # H0 true
    decision, n = tester.run_test(data)
    decisions_h0.append(decision)
    n_obs_h0.append(n)

# Type I error = fraction where decision is 'reject_H0'
type_i_error = np.mean(np.array(decisions_h0) == 'reject_H0')
```

</details>

In [ ]:
# YOUR CODE HERE
#
# 1. Under H0: Verify Type I error control
# 2. Under H1: Verify power (1 - Type II error)
# 3. Report average stopping time vs fixed sample size

n_simulations = 1000
max_days = 60
obs_per_day = 50
alpha = 0.05
beta = 0.20

# YOUR CODE HERE
pass

In [ ]:
# Verification (uncomment after completing):

# print("=== Under H0 (no effect) ===")
# print(f"Type I error rate: {type_i_error:.3f} (target: <= {alpha})")
# print(f"Average stopping time: {np.mean(n_obs_h0):.0f} observations")

# print(f"\n=== Under H1 (effect = {effect_size}) ===")
# print(f"Power (detection rate): {power:.3f} (target: >= {1-beta})")
# print(f"Average stopping time: {np.mean(n_obs_h1):.0f} observations")

# # Compare to fixed-horizon sample size
# from scipy.stats import norm
# z_a = norm.ppf(1 - alpha/2)
# z_b = norm.ppf(1 - beta)
# n_fixed = int(np.ceil(((z_a + z_b) * sigma / effect_size)**2))
# print(f"\nFixed-horizon sample size: {n_fixed}")
# print(f"SPRT average (H1): {np.mean(n_obs_h1):.0f} ({np.mean(n_obs_h1)/n_fixed*100:.0f}% of fixed)")

# # Verify error control
# assert type_i_error <= alpha + 0.02, f"Type I error {type_i_error:.3f} exceeds alpha={alpha}"
# assert power >= (1 - beta) - 0.05, f"Power {power:.3f} below target {1-beta}"
# print("\nTask 3 checks passed! SPRT controls error rates correctly.")

---

## Task 4: Visualize the SPRT Path

### Context

A powerful way to communicate sequential testing to stakeholders is to show the **decision path**: how the cumulative log-likelihood ratio evolves over time, with the decision boundaries clearly marked.

### Problem

Create a visualization that shows:
1. The cumulative log-likelihood ratio over time (one observation at a time)
2. The upper boundary (horizontal line) labeled "reject H0"
3. The lower boundary (horizontal line) labeled "reject H1"
4. The point where the boundary is crossed (decision point)

Create two plots side by side:
- Left: A test under H1 (real effect) — should cross the upper boundary
- Right: A test under H0 (no effect) — should cross the lower boundary

<details>
<summary>Hint</summary>

Run the tester and use its `history` attribute (cumulative sum at each step):

```python
tester = SequentialTester(effect_size=0.3, sigma=1.0)
data = np.random.normal(0.3, 1.0, 500)  # under H1
tester.run_test(data)

# tester.history contains the cumulative log-LR at each observation
plt.plot(range(1, len(tester.history) + 1), tester.history)
plt.axhline(tester.upper_bound, color='green', linestyle='--', label='Reject H0')
plt.axhline(tester.lower_bound, color='red', linestyle='--', label='Reject H1')
```

</details>

In [ ]:
# YOUR CODE HERE
#
# Create side-by-side plots showing SPRT paths:
# Left: under H1 (real effect)
# Right: under H0 (no effect)
#
# Each plot should show:
# - Cumulative log-LR path (blue line)
# - Upper boundary (green dashed)
# - Lower boundary (red dashed)
# - Decision point marked (star or dot)
# - Shaded "continue" region between boundaries

# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
# ...
pass

**Your interpretation:**

*Describe what you observe in the two plots. How does the log-LR path behave differently under H0 vs H1? Why does SPRT stop early when there's a real effect?*

---

## Summary

In this assignment you:
- Demonstrated that naive peeking inflates false positive rates 4-5x
- Implemented Wald's SPRT with proper decision boundaries
- Verified that SPRT controls both Type I and Type II error rates
- Showed that SPRT reaches decisions faster than fixed-horizon tests
- Visualized the sequential decision process

**Key takeaways:**
1. Never peek at fixed-horizon tests repeatedly — use sequential methods if you want early stopping.
2. SPRT provides a principled framework: specify the errors you'll tolerate ($\alpha$, $\beta$) and the effect size you care about, and the test tells you when to stop.
3. The trade-off: SPRT requires specifying H1 in advance (what effect size to power for), and it can take longer than fixed-horizon when the true effect is much smaller than specified.